In [46]:
import json
import os
import time
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from typing import List, Dict

In [47]:

EMBEDDING_DIR  = "../embedding/output"
BENCHMARK_PATH = "../data/benchmark_dataset.json"
RESULTS_DIR    = "results/"
os.makedirs(RESULTS_DIR, exist_ok=True)
CHUNK_STRATEGY = "chunks_fixed300_overlap20"
EMBED_MODEL    = "minilm"                       # "minilm" | "nomic"
USE_RERANKER   = True                           # True = bi-encoder + reranker; False = bi-encoder only
TOP_K          = 5
CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# -- Model registry --
MODEL_REGISTRY = {
    "minilm": {
        "model_name": "all-MiniLM-L6-v2",
        "query_prefix": "",
    },
    "nomic": {
        "model_name": "nomic-ai/nomic-embed-text-v1.5",
        "trust_remote_code": True,
        "query_prefix": "search_query: ",
    },
}

# -- Derived paths --
FAISS_PATH = os.path.join(EMBEDDING_DIR, CHUNK_STRATEGY, f"embedded_{EMBED_MODEL}.faiss")
JSONL_PATH = os.path.join(EMBEDDING_DIR, CHUNK_STRATEGY, f"embedded_{EMBED_MODEL}.jsonl")

print(f"Chunk strategy : {CHUNK_STRATEGY}")
print(f"Embed model    : {EMBED_MODEL} -> {MODEL_REGISTRY[EMBED_MODEL]['model_name']}")
print(f"Reranker       : {'ON' if USE_RERANKER else 'OFF'}")
print(f"Benchmark      : {BENCHMARK_PATH}")
print(f"FAISS index    : {FAISS_PATH}")



Chunk strategy : chunks_fixed300_overlap20
Embed model    : minilm -> all-MiniLM-L6-v2
Reranker       : ON
Benchmark      : ../data/benchmark_dataset.json
FAISS index    : ../embedding/output\chunks_fixed300_overlap20\embedded_minilm.faiss


In [48]:
def load_benchmark(path: str) -> List[Dict]:
    """Load benchmark JSON (nested source-grouped format):
    {"sources": [{"source": "...", "questions": [{"question": ..., "answer": ...}]}]}
    Returns flat list with keys: query_id, query, answer, source.
    """
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    flat = []
    idx = 1
    for src_block in data["sources"]:
        source = src_block["source"]
        for q in src_block["questions"]:
            flat.append({
                "query_id": f"q_{idx:03d}",
                "query":    q["question"],
                "answer":   q["answer"],
                "source":   source,
            })
            idx += 1
    return flat


def load_encoder(embed_model: str) -> SentenceTransformer:
    """Load the bi-encoder for the given model tag."""
    config = MODEL_REGISTRY[embed_model]
    kwargs = {}
    if config.get("trust_remote_code"):
        kwargs["trust_remote_code"] = True
    model = SentenceTransformer(config["model_name"], **kwargs)
    print(f"Bi-encoder loaded: {config['model_name']}")
    return model


def encode_queries(queries: List[str], embed_model: str, encoder: SentenceTransformer) -> np.ndarray:
    """Encode query texts with appropriate prefix for the model."""
    prefix = MODEL_REGISTRY[embed_model].get("query_prefix", "")
    texts = [prefix + q for q in queries] if prefix else queries
    return encoder.encode(
        texts, normalize_embeddings=True, show_progress_bar=True
    ).astype(np.float32)


def retrieve_biencoder(
    queries: List[Dict],
    index: faiss.Index,
    corpus_lines: List[str],
    encoder: SentenceTransformer,
    embed_model: str,
    top_n: int,
) -> List[Dict]:
    """Bi-encoder retrieval. Returns top-n candidates per query."""
    query_texts = [q["query"] for q in queries]
    query_embs = encode_queries(query_texts, embed_model, encoder)
    scores_all, indices_all = index.search(query_embs, top_n)

    results = []
    for q, scores, indices in zip(queries, scores_all, indices_all):
        retrieved = []
        for rank, (score, idx) in enumerate(zip(scores, indices), start=1):
            if idx == -1:
                continue
            chunk = json.loads(corpus_lines[idx])
            retrieved.append({
                "rank": rank,
                "chunk_id": chunk["chunk_id"],
                "score": round(float(score), 6),
                "text": chunk["text"],
            })
        results.append({
            "query_id": q["query_id"],
            "query": q["query"],
            "retrieved": retrieved,
        })
    return results


def rerank(
    stage1_results: List[Dict],
    cross_encoder: CrossEncoder,
    top_k: int = 5,
) -> List[Dict]:
    """Re-rank Stage 1 candidates with a Cross-Encoder. Returns top-k."""
    reranked = []
    for item in stage1_results:
        query = item["query"]
        candidates = item["retrieved"]
        if not candidates:
            reranked.append(item)
            continue

        pairs = [(query, c["text"]) for c in candidates]
        ce_scores = cross_encoder.predict(pairs)
        scored = sorted(zip(ce_scores, candidates), key=lambda x: x[0], reverse=True)

        top_k_hits = []
        for new_rank, (ce_score, cand) in enumerate(scored[:top_k], start=1):
            top_k_hits.append({
                "rank":         new_rank,
                "chunk_id":     cand["chunk_id"],
                "score":        cand["score"],
                "rerank_score": round(float(ce_score), 6),
                "text":         cand["text"],
            })
        reranked.append({
            "query_id":  item["query_id"],
            "query":     query,
            "retrieved": top_k_hits,
        })
    return reranked


def save_results(results: List[Dict], path: str):
    with open(path, "w", encoding="utf-8") as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"  Saved {len(results)} queries -> {path}")


## Stage 1: Single-config Demo


In [49]:
# Load resources for the default config
benchmark = load_benchmark(BENCHMARK_PATH)
print(f"Loaded {len(benchmark)} benchmark queries")

index = faiss.read_index(FAISS_PATH)
with open(JSONL_PATH, "r", encoding="utf-8") as f:
    corpus_lines = f.readlines()
assert len(corpus_lines) == index.ntotal
print(f"FAISS index: {index.ntotal} vectors, dim={index.d}")

bi_encoder = load_encoder(EMBED_MODEL)

Loaded 72 benchmark queries
FAISS index: 2253 vectors, dim=384


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Bi-encoder loaded: all-MiniLM-L6-v2


In [50]:
# Stage 1A: Top-5 standalone
BENCH_NAME = os.path.splitext(os.path.basename(BENCHMARK_PATH))[0]
t0 = time.time()
results_top5 = retrieve_biencoder(benchmark, index, corpus_lines, bi_encoder, EMBED_MODEL, top_n=5)
print(f"Top-5 done: {len(results_top5)} queries, {time.time()-t0:.2f}s")

top5_path = os.path.join(RESULTS_DIR, f"retrieval_biencoder_top5_{EMBED_MODEL}_{CHUNK_STRATEGY}_{BENCH_NAME}.jsonl")
save_results(results_top5, top5_path)

# Inspect first query
sample = results_top5[0]
print(f"\nQuery: {sample['query']}")
for r in sample["retrieved"]:
    print(f"  #{r['rank']}  score={r['score']:.4f}  {r['chunk_id']}")
    print(f"       {r['text'][:90]}...")


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Top-5 done: 72 queries, 0.35s
  Saved 72 queries -> results/retrieval_biencoder_top5_minilm_chunks_fixed300_overlap20_benchmark_dataset.jsonl

Query: What does the word 'bibimbap' literally mean in Korean?
  #1  score=0.6908  wiki_0006_chunk_0
       Bibimbap ( BEE-bim-bap; Korean: 비빔밥; lit. 'mixed rice'), sometimes romanised as bi bim bap...
  #2  score=0.6098  wiki_0006_chunk_6
       th century caused a cultural divide in the creation of bibimbap, with two types related to...
  #3  score=0.6043  wiki_0006_chunk_5
       of all the leftover side dishes before the new year. The solution to this problem was to p...
  #4  score=0.5976  wiki_0006_chunk_2
       encyclopedia Myeongmul giryak states that the dish name is written as goldong-ban (骨董飯) in...
  #5  score=0.5887  wiki_0006_chunk_4
       and use of bibimbap. The first one, "People's Unofficial Story of Jeonju" (全州野史), describe...


In [51]:
# Stage 1B: Top-20 (only needed when using reranker)
if USE_RERANKER:
    t0 = time.time()
    results_top20 = retrieve_biencoder(benchmark, index, corpus_lines, bi_encoder, EMBED_MODEL, top_n=20)
    print(f"Top-20 done: {len(results_top20)} queries, {time.time()-t0:.2f}s")

    top20_path = os.path.join(RESULTS_DIR, f"retrieval_biencoder_top20_{EMBED_MODEL}_{CHUNK_STRATEGY}_{BENCH_NAME}.jsonl")
    save_results(results_top20, top20_path)
else:
    print("Reranker disabled -- skipping top-20 retrieval")


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Top-20 done: 72 queries, 0.58s
  Saved 72 queries -> results/retrieval_biencoder_top20_minilm_chunks_fixed300_overlap20_benchmark_dataset.jsonl



## Stage 2: Cross-Encoder Reranking (optional)


In [52]:
if USE_RERANKER:
    cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL)
    print(f"Cross-encoder loaded: {CROSS_ENCODER_MODEL}")

    t0 = time.time()
    results_reranked = rerank(results_top20, cross_encoder, top_k=TOP_K)
    print(f"Reranking done: {len(results_reranked)} queries, {time.time()-t0:.2f}s")

    reranked_path = os.path.join(RESULTS_DIR, f"retrieval_reranked_top5_{EMBED_MODEL}_{CHUNK_STRATEGY}_{BENCH_NAME}.jsonl")
    save_results(results_reranked, reranked_path)
else:
    print("Reranker disabled -- skipping Stage 2")
    results_reranked = None


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Cross-encoder loaded: cross-encoder/ms-marco-MiniLM-L-6-v2
Reranking done: 72 queries, 62.56s
  Saved 72 queries -> results/retrieval_reranked_top5_minilm_chunks_fixed300_overlap20_benchmark_dataset.jsonl


In [53]:
# Inspect reranked sample
if results_reranked:
    sample = results_reranked[0]
    print(f"Query: {sample['query']}\n")
    for r in sample["retrieved"]:
        print(f"  #{r['rank']}  bi={r['score']:.4f}  ce={r['rerank_score']:.4f}  {r['chunk_id']}")
        print(f"       {r['text'][:90]}...\n")


Query: What does the word 'bibimbap' literally mean in Korean?

  #1  bi=0.5826  ce=7.2275  wikibooks_0069_chunk_0
       Bibimbap (also called Bibimba or Bibimbop) is a typical Korean meal that literally means "...

  #2  bi=0.6908  ce=6.8768  wiki_0006_chunk_0
       Bibimbap ( BEE-bim-bap; Korean: 비빔밥; lit. 'mixed rice'), sometimes romanised as bi bim bap...

  #3  bi=0.4945  ce=5.8916  blog_0022_chunk_0
       Bibimbap literally means “mixed rice” and describes a dish of cooked white rice topped wit...

  #4  bi=0.5112  ce=5.5879  wiki_0006_chunk_9
       and popular bibimbap made it one of the most popular foods in Jecheon.

Hoedeopbap (회덮밥)
H...

  #5  bi=0.5976  ce=4.3011  wiki_0006_chunk_2
       encyclopedia Myeongmul giryak states that the dish name is written as goldong-ban (骨董飯) in...



In [54]:
# Rank-change analysis: how much does the cross-encoder reorder?
if results_reranked:
    top1_changed = 0
    rank_shifts = []

    for s1, s2 in zip(results_top5, results_reranked):
        s1_ids = [h["chunk_id"] for h in s1["retrieved"]]
        s2_ids = [h["chunk_id"] for h in s2["retrieved"]]
        if s1_ids and s2_ids and s1_ids[0] != s2_ids[0]:
            top1_changed += 1
        for new_rank, chunk_id in enumerate(s2_ids, start=1):
            if chunk_id in s1_ids:
                old_rank = s1_ids.index(chunk_id) + 1
                rank_shifts.append(abs(new_rank - old_rank))

    n = len(results_top5)
    avg_shift = sum(rank_shifts) / max(len(rank_shifts), 1)
    print(f"Top-1 changed by reranking : {top1_changed}/{n} ({top1_changed/n*100:.0f}%)")
    print(f"Avg absolute rank shift    : {avg_shift:.2f} positions")
else:
    print("No reranking to analyse")


Top-1 changed by reranking : 25/72 (35%)
Avg absolute rank shift    : 0.93 positions


Exploration -- All Combinations

In [55]:
# Define the exploration grid
CHUNK_STRATEGIES = [
    "chunks_fixed300_overlap0",
    "chunks_fixed300_overlap20",
    "chunks_sentence220_overlap1",
    "chunks_sentence220_overlap2",
    "chunks_structure_aware_coarse",
    "chunks_structure_aware_fine",
]

EMBED_MODELS = ["minilm", "nomic"]

BENCHMARK_PATHS = {
    "benchmark_dataset":   "../data/benchmark_dataset.json",
}

def get_result_path(retrieval_type, chunk_strategy, embed_model, bench_name):
    return os.path.join(
        RESULTS_DIR,
        f"retrieval_{retrieval_type}_{embed_model}_{chunk_strategy}_{bench_name}.jsonl"
    )

total = len(CHUNK_STRATEGIES) * len(EMBED_MODELS) * len(BENCHMARK_PATHS)
print(f"Exploration grid: {len(CHUNK_STRATEGIES)} chunk x {len(EMBED_MODELS)} embed x {len(BENCHMARK_PATHS)} bench = {total} combos")
print(f"Each combo -> 3 files (top5, top20, reranked) = {total * 3} result files total")


Exploration grid: 6 chunk x 2 embed x 1 bench = 12 combos
Each combo -> 3 files (top5, top20, reranked) = 36 result files total


In [56]:
# Cache to avoid reloading models
_encoder_cache = {}
_cross_encoder = None

def get_encoder_cached(embed_model: str) -> SentenceTransformer:
    if embed_model not in _encoder_cache:
        _encoder_cache[embed_model] = load_encoder(embed_model)
    return _encoder_cache[embed_model]

def get_cross_encoder_cached() -> CrossEncoder:
    global _cross_encoder
    if _cross_encoder is None:
        _cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL)
        print(f"Cross-encoder loaded: {CROSS_ENCODER_MODEL}")
    return _cross_encoder

In [57]:
# Run exploration
exploration_log = []
run_idx = 0

for bench_name, bench_path in BENCHMARK_PATHS.items():
    if not os.path.exists(bench_path):
        print(f"[SKIP] benchmark not found: {bench_path}")
        continue
    bench_data = load_benchmark(bench_path)

    for embed_model in EMBED_MODELS:
        encoder = get_encoder_cached(embed_model)

        for chunk_strat in CHUNK_STRATEGIES:
            run_idx += 1
            tag = f"{bench_name} | {chunk_strat} | {embed_model}"

            f_path = os.path.join(EMBEDDING_DIR, chunk_strat, f"embedded_{embed_model}.faiss")
            j_path = os.path.join(EMBEDDING_DIR, chunk_strat, f"embedded_{embed_model}.jsonl")
            if not os.path.exists(f_path):
                print(f"[{run_idx}/{total}] SKIP (no FAISS): {tag}")
                continue

            print(f"\n[{run_idx}/{total}] {tag}")
            t0 = time.time()

            idx = faiss.read_index(f_path)
            with open(j_path, "r", encoding="utf-8") as f:
                c_lines = f.readlines()

            # biencoder top-5
            res_top5 = retrieve_biencoder(bench_data, idx, c_lines, encoder, embed_model, top_n=5)
            save_results(res_top5, get_result_path("biencoder_top5", chunk_strat, embed_model, bench_name))

            # biencoder top-20 + rerank
            res_top20 = retrieve_biencoder(bench_data, idx, c_lines, encoder, embed_model, top_n=20)
            save_results(res_top20, get_result_path("biencoder_top20", chunk_strat, embed_model, bench_name))

            ce = get_cross_encoder_cached()
            res_reranked = rerank(res_top20, ce, top_k=TOP_K)
            save_results(res_reranked, get_result_path("reranked_top5", chunk_strat, embed_model, bench_name))

            elapsed = time.time() - t0
            exploration_log.append({
                "benchmark": bench_name,
                "chunking": chunk_strat,
                "embedding": embed_model,
                "n_chunks": idx.ntotal,
                "time_s": round(elapsed, 2),
            })
            print(f"  Done in {elapsed:.1f}s  ({idx.ntotal} chunks)")

log_path = os.path.join(RESULTS_DIR, "exploration_retrieval_log.json")
with open(log_path, "w", encoding="utf-8") as f:
    json.dump(exploration_log, f, indent=2)
print(f"\nExploration complete: {len(exploration_log)} combos. Log -> {log_path}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Bi-encoder loaded: all-MiniLM-L6-v2

[1/12] benchmark_dataset | chunks_fixed300_overlap0 | minilm


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

  Saved 72 queries -> results/retrieval_biencoder_top5_minilm_chunks_fixed300_overlap0_benchmark_dataset.jsonl


...

  Saved 72 queries -> results/retrieval_biencoder_top20_nomic_chunks_structure_aware_coarse_benchmark_dataset.jsonl
  Saved 72 queries -> results/retrieval_reranked_top5_nomic_chunks_structure_aware_coarse_benchmark_dataset.jsonl
  Done in 91.1s  (2607 chunks)

[12/12] benchmark_dataset | chunks_structure_aware_fine | nomic


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

  Saved 72 queries -> results/retrieval_biencoder_top5_nomic_chunks_structure_aware_fine_benchmark_dataset.jsonl


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

  Saved 72 queries -> results/retrieval_biencoder_top20_nomic_chunks_structure_aware_fine_benchmark_dataset.jsonl
  Saved 72 queries -> results/retrieval_reranked_top5_nomic_chunks_structure_aware_fine_benchmark_dataset.jsonl
  Done in 91.5s  (2444 chunks)

Exploration complete: 12 combos. Log -> results/exploration_retrieval_log.json


In [59]:
# Summary table
print(f"{'Benchmark':20s} {'Chunking':32s} {'Embed':8s} {'Chunks':>6s} {'Time':>6s}")
print('-' * 80)
for entry in exploration_log:
    print(
        f"{entry['benchmark']:20s} "
        f"{entry['chunking']:32s} "
        f"{entry['embedding']:8s} "
        f"{entry['n_chunks']:>6d} "
        f"{entry['time_s']:>5.1f}s"
    )

Benchmark            Chunking                         Embed    Chunks   Time
--------------------------------------------------------------------------------
benchmark_dataset    chunks_fixed300_overlap0         minilm     1838  79.0s
benchmark_dataset    chunks_fixed300_overlap20        minilm     2253  68.0s
benchmark_dataset    chunks_sentence220_overlap1      minilm     2145  85.6s
benchmark_dataset    chunks_sentence220_overlap2      minilm     2394  85.9s
benchmark_dataset    chunks_structure_aware_coarse    minilm     2607  84.2s
benchmark_dataset    chunks_structure_aware_fine      minilm     2444  84.2s
benchmark_dataset    chunks_fixed300_overlap0         nomic      1838  81.8s
benchmark_dataset    chunks_fixed300_overlap20        nomic      2253  74.0s
benchmark_dataset    chunks_sentence220_overlap1      nomic      2145  81.4s
benchmark_dataset    chunks_sentence220_overlap2      nomic      2394  82.3s
benchmark_dataset    chunks_structure_aware_coarse    nomic      2607  9